# 07 · Quantile Regression + EVT — Finance Concept

**Contexto:** Las pérdidas económicas por desastres naturales tienen colas extremadamente pesadas — el percentil 99% puede ser 10-50x la media. Los modelos de riesgo basados en Normal subestiman sistemáticamente el riesgo en los percentiles extremos. Quantile Regression y EVT modelan directamente esas colas sin asumir distribución.

**Campo de origen:** Econometría (Koenker & Bassett 1978) + Estadística de extremos (Gumbel 1935, Balkema-de Haan-Pickands 1974)  
**Dataset:** INDECI — Sistema Nacional de Información para la Gestión del Riesgo de Desastres (SINPAD)  
**Fuente real:** https://sinpad.indeci.gob.pe/  
**Contexto:** Perú es uno de los países con mayor exposición a desastres naturales del mundo (huaycos, inundaciones, sismos, sequías). Las pérdidas económicas tienen distribución Pareto-like con $\xi > 0$.

---

## Marco teórico

### Quantile Regression — Koenker & Bassett (1978)

$$\hat{\beta}_\tau = \arg\min_\beta \sum_{i=1}^n \rho_\tau(y_i - x_i\beta) \qquad \rho_\tau(u) = u(\tau - \mathbf{1}_{u<0})$$

### Peaks Over Threshold (POT) — Balkema-de Haan-Pickands

$$P(X - u \leq y \mid X > u) \xrightarrow{u \to \infty} H(y; \sigma, \xi) = 1 - \left(1 + \frac{\xi y}{\sigma}\right)^{-1/\xi}$$

### Cuantil extremo (nivel de retorno)

$$Q_p = u + \frac{\hat{\sigma}}{\hat{\xi}}\left[\left(\frac{n}{n_u \cdot p}\right)^{\hat{\xi}} - 1\right]$$

**Referencias:** Koenker & Bassett (1978). *Econometrica* 46(1). Coles, S. (2001). *Introduction to Statistical Modeling of Extreme Values*. Springer. INDECI (2024). SINPAD — Estadísticas de Emergencias.

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    loss='#DC2626',   qr='#2563EB',    evt='#15803D',
    normal='#94A3B8', fill='#FEE2E2',  gpd='#7C3AED',
    mep='#F59E0B',    threshold='#1E293B'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — INDECI SINPAD: Emergencias y pérdidas económicas Perú ─────────────
# Fuente real: INDECI — Sistema Nacional de Información para la Gestión
#              del Riesgo de Desastres (SINPAD)
# URL: https://sinpad.indeci.gob.pe/
# Datos: emergencias anuales por tipo (huayco, inundación, sismo, sequía)
#        pérdidas económicas en millones de soles por departamento
#
# Estadísticos reales INDECI 2000-2023 documentados:
#   n eventos/año promedio: ~3,200  ·  pérdida media: S/.12M/evento grande
#   Distribución: Pareto-like con ξ ≈ 0.45-0.65 (cola pesada confirmada)
#   Eventos extremos: El Niño 1997-98 (~S/.4,500M), El Niño Costero 2017 (~S/.3,100M)
#   Departamentos más afectados: Piura, Loreto, Cusco, Puno
#   Estacionalidad: pico Feb-Abr (temporada lluvias sierra y selva)
#
# Simulación calibrada con estadísticos reales INDECI

SOURCE = 'Simulación calibrada con estadísticos reales INDECI SINPAD 2000-2023'

n_eventos = 500  # eventos grandes (pérdidas > S/.100K)
np.random.seed(42)

# Pérdidas: mezcla de log-normal (eventos menores) + Pareto (eventos extremos)
# Pareto con ξ=0.55 es consistente con literatura de desastres naturales Perú
n_normal  = int(n_eventos * 0.92)  # 92% eventos moderados
n_extreme = n_eventos - n_normal   # 8% eventos extremos (El Niño, sismos mayores)

# Pérdidas moderadas: log-normal con μ=13.5, σ=1.0 (~S/.0.7M media)
losses_mod  = np.random.lognormal(mean=13.5, sigma=1.0, size=n_normal) / 1e6  # en M soles

# Pérdidas extremas: Pareto generalizada con ξ=0.55, u=5M soles
u_thresh = 5.0  # umbral en M soles
xi_true, sigma_true = 0.55, 8.0
# Simular GPD: F^{-1}(u) = u + σ/ξ·[(1-u)^{-ξ} - 1]
uniform_ext = np.random.uniform(0, 1, n_extreme)
losses_ext  = u_thresh + sigma_true/xi_true * ((1-uniform_ext)**(-xi_true) - 1)

# Covariables
tipo_evento = np.array(
    ['huayco'] * (n_normal//3) +
    ['inundacion'] * (n_normal//3) +
    ['sismo_sequia'] * (n_normal - 2*(n_normal//3)) +
    ['el_nino'] * n_extreme
)
mes = np.random.choice(['dic_abr', 'may_nov'], size=n_eventos,
                        p=[0.65, 0.35])  # pico en temporada lluvias

all_losses = np.concatenate([losses_mod, losses_ext])
# Añadir ruido
all_losses = all_losses * np.random.lognormal(0, 0.15, n_eventos)
all_losses = np.maximum(all_losses, 0.1)

df = pd.DataFrame({
    'perdida_mS': all_losses,
    'log_perdida': np.log(all_losses),
    'tipo': tipo_evento,
    'temporada_lluvia': (mes == 'dic_abr').astype(int),
    'es_el_nino': (tipo_evento == 'el_nino').astype(int),
})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Fuente : {SOURCE}')
print(f'n      : {len(df)} eventos registrados')
print(f'\n── Pérdidas (M soles) ───────────────────────────────────────')
for stat, val in [
    ('Media',        df.perdida_mS.mean()),
    ('Mediana',      df.perdida_mS.median()),
    ('p90',          df.perdida_mS.quantile(.90)),
    ('p95',          df.perdida_mS.quantile(.95)),
    ('p99',          df.perdida_mS.quantile(.99)),
    ('Máximo',       df.perdida_mS.max()),
    ('Skewness',     df.perdida_mS.skew()),
    ('Kurtosis',     df.perdida_mS.kurt()),
]:
    print(f'  {stat:<12}: {val:.2f}')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Distribución y colas ───────────────────────────────────────────
x = df.perdida_mS.values
mu, sigma = x.mean(), x.std()

print(f'{"Métrica":<20} {"Valor":>12}  Nota')
print('─' * 60)
for label, val, note in [
    ('n eventos',      len(x),              ''),
    ('Media (M S/.)',  f'{mu:.2f}',          ''),
    ('Mediana (M S/.)',f'{np.median(x):.2f}', 'media >> mediana → cola derecha'),
    ('σ (M S/.)',      f'{sigma:.2f}',        ''),
    ('CV',             f'{sigma/mu:.3f}',     '> 1 → muy disperso'),
    ('Skewness',       f'{stats.skew(x):.2f}','> 3 → cola pesada extrema'),
    ('Kurtosis',       f'{stats.kurtosis(x):.2f}', '> 10 → fat tail severo'),
    ('p99 / media',    f'{np.percentile(x,99)/mu:.1f}x','ratio cola extrema'),
    ('Max / p99',      f'{x.max()/np.percentile(x,99):.1f}x', 'El Niño vs. percentil 99'),
]:
    print(f'{label:<20} {str(val):>12}  {note}')

In [ ]:
# ── EDA 2/2 — Histograma + QQ-plot ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    'Mini-EDA — INDECI: Pérdidas económicas por desastres naturales Perú\n'
    f'{SOURCE}', fontsize=10, y=1.01
)

# Histograma log-escala
ax = axes[0]
ax.hist(np.log10(x+0.01), bins=40, color=C['fill'],
        edgecolor=C['loss'], lw=0.4, density=True)
ax.set_xlabel('log₁₀(Pérdida M S/.)'); ax.set_ylabel('Densidad')
ax.set_title('Distribución log₁₀ pérdidas\n(aproximadamente log-normal en cuerpo)', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# QQ-plot vs. Normal
ax2 = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(x, dist='norm', plot=None)
ax2.scatter(osm, osr, color=C['loss'], s=8, alpha=0.5)
ax2.plot(osm, slope*np.array(osm)+intercept, color=C['normal'], lw=1.2, label='Normal')
ax2.set_xlabel('Cuantiles teóricos Normal')
ax2.set_ylabel('Cuantiles observados')
ax2.set_title('QQ-plot vs. Normal\n(cola derecha muy por encima → fat tail)', fontsize=9)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# Pérdidas por tipo
ax3 = axes[2]
tipos = ['huayco', 'inundacion', 'sismo_sequia', 'el_nino']
medias = [df[df.tipo==t].perdida_mS.mean() for t in tipos]
p95s   = [df[df.tipo==t].perdida_mS.quantile(.95) for t in tipos]
x_pos  = np.arange(len(tipos))
ax3.bar(x_pos, medias, color=C['normal'], alpha=0.7, label='Media')
ax3.bar(x_pos, p95s,   color=C['loss'],   alpha=0.5, label='p95')
ax3.set_xticks(x_pos); ax3.set_xticklabels(['Huayco','Inundación','Sismo/Sequía','El Niño'],
                                             fontsize=8)
ax3.set_ylabel('Pérdida (M S/.)'); ax3.set_title('Media vs. p95 por tipo\n(El Niño domina la cola)', fontsize=9)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/finance_eda.png')

In [ ]:
# ── QUANTILE REGRESSION ───────────────────────────────────────────────────────

def check_loss(u, tau):
    """Función de pérdida check para quantile regression."""
    return np.where(u >= 0, tau * u, (tau - 1) * u).sum()


def quantile_regression(X, y, tau, n_iter=2000, lr=0.001):
    """
    QR por subgradiente (implementación desde cero).
    X: array (n, p+1) con columna de unos
    y: array (n,)
    tau: cuantil objetivo ∈ (0, 1)
    """
    n, p = X.shape
    beta = np.zeros(p)
    best_beta = beta.copy()
    best_loss = np.inf

    for it in range(n_iter):
        # Ajustar lr
        lr_t = lr / (1 + 0.001 * it)
        resid = y - X @ beta
        # Subgradiente de la función check
        grad = X.T @ np.where(resid >= 0, -tau, 1 - tau) / n
        beta = beta - lr_t * grad
        loss = check_loss(resid, tau)
        if loss < best_loss:
            best_loss = loss
            best_beta = beta.copy()

    return best_beta


# Preparar datos: log(pérdida) ~ 1 + temporada_lluvia + es_el_nino
y_qr = df.log_perdida.values
X_qr = np.column_stack([
    np.ones(len(df)),
    df.temporada_lluvia.values,
    df.es_el_nino.values
])

# Estimar QR para múltiples cuantiles
taus = [0.50, 0.75, 0.90, 0.95, 0.99]
betas_qr = {}
for tau in taus:
    b = quantile_regression(X_qr, y_qr, tau, n_iter=3000, lr=0.005)
    betas_qr[tau] = b

print('── Quantile Regression sobre log(pérdida) ───────────────────────')
print(f'{"τ":<8} {"β₀ (base)":>12} {"β₁ (lluvia)":>12} {"β₂ (El Niño)":>14}')
print('─' * 50)
for tau, b in betas_qr.items():
    print(f'{tau:<8.2f} {b[0]:>12.3f} {b[1]:>12.3f} {b[2]:>14.3f}')

print('\n── Interpretación ───────────────────────────────────────────────')
print('β₁ (lluvia): impacto de temporada de lluvias en cada cuantil')
print('β₂ (El Niño): impacto de El Niño en cada cuantil (en log M S/.)')
print('→ Si β₂ crece con τ: El Niño impacta más los eventos extremos que la media')

In [ ]:
# ── EVT — PEAKS OVER THRESHOLD ───────────────────────────────────────────────

def mean_excess_plot(x, u_range=None, n_points=30):
    """Calcula la función de exceso medio e(u) para distintos umbrales."""
    x = np.sort(x)
    if u_range is None:
        u_range = np.linspace(np.percentile(x, 60), np.percentile(x, 97), n_points)
    e_u, n_u = [], []
    for u in u_range:
        excesos = x[x > u] - u
        if len(excesos) >= 5:
            e_u.append(excesos.mean())
            n_u.append(len(excesos))
    return np.array(u_range[:len(e_u)]), np.array(e_u), np.array(n_u)


def fit_gpd_mle(excesos):
    """
    Ajusta GPD por MLE a los excesos sobre el umbral.
    H(y; σ, ξ) = 1 - (1 + ξy/σ)^{-1/ξ}
    """
    def neg_ll(params):
        sigma, xi = params
        if sigma <= 0: return 1e10
        if xi == 0:
            return np.sum(np.log(sigma) + excesos/sigma)
        arg = 1 + xi * excesos / sigma
        if np.any(arg <= 0): return 1e10
        return np.sum(np.log(sigma) + (1/xi + 1)*np.log(arg))

    # Inicializar con estimador de momentos
    m1 = excesos.mean()
    m2 = excesos.var()
    xi0 = 0.5 * (1 - m1**2/m2)
    s0  = 0.5 * m1 * (1 + m1**2/m2)
    s0  = max(s0, 0.01)

    res = minimize(neg_ll, [s0, max(xi0, 0.01)],
                   method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-6})
    return res.x[0], res.x[1]  # sigma_hat, xi_hat


def gpd_quantile(u, sigma, xi, n, n_u, p):
    """Cuantil extremo Q_p = u + σ/ξ·[(n/(n_u·p))^ξ - 1]"""
    if xi == 0:
        return u - sigma * np.log(n_u * p / n)
    return u + sigma/xi * ((n / (n_u * p))**xi - 1)


# Elegir umbral u = p80 de pérdidas
u_thresh = np.percentile(df.perdida_mS, 80)
excesos  = df.perdida_mS[df.perdida_mS > u_thresh].values - u_thresh
n_total  = len(df)
n_excess = len(excesos)

sigma_hat, xi_hat = fit_gpd_mle(excesos)

print(f'── EVT / POT ────────────────────────────────────────────────────')
print(f'  Umbral u   : {u_thresh:.2f} M S/. (p80)')
print(f'  n excesos  : {n_excess} de {n_total} ({n_excess/n_total:.1%})')
print(f'  σ̂ (GPD)   : {sigma_hat:.4f}')
print(f'  ξ̂ (GPD)   : {xi_hat:.4f}  '
      f'({"cola pesada (Fréchet)" if xi_hat > 0 else "cola acotada (Weibull)"})')

print(f'\n── Cuantiles extremos ───────────────────────────────────────────')
print(f'{"Cuantil":<12} {"GPD (M S/.)":>14} {"Normal (M S/.)":>16} {"Ratio GPD/Normal":>18}')
print('─' * 64)
for p_exc in [0.10, 0.05, 0.01, 0.005, 0.001]:
    tau_q = 1 - p_exc
    q_gpd = gpd_quantile(u_thresh, sigma_hat, xi_hat, n_total, n_excess, p_exc)
    q_norm = df.perdida_mS.mean() + stats.norm.ppf(tau_q) * df.perdida_mS.std()
    ratio = q_gpd / q_norm if q_norm > 0 else np.nan
    print(f'  p{tau_q*100:.1f}%  {q_gpd:>14.1f} {q_norm:>16.1f} {ratio:>18.2f}x')

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 13))
fig.suptitle(
    'Quantile Regression + EVT — INDECI Perú\n'
    'Pérdidas económicas por desastres naturales (huaycos, inundaciones, El Niño)',
    fontsize=12, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(3, 2, hspace=0.42, wspace=0.30)

# P1 — Quantile regression: coeficientes por τ
ax1 = fig.add_subplot(gs[0, 0])
taus_plot = list(betas_qr.keys())
b1s = [betas_qr[t][1] for t in taus_plot]  # coef. temporada lluvia
b2s = [betas_qr[t][2] for t in taus_plot]  # coef. El Niño
ax1.plot(taus_plot, b1s, 'o-', color=C['qr'],  lw=1.5, label='β₁ (temporada lluvia)')
ax1.plot(taus_plot, b2s, 's-', color=C['loss'], lw=1.5, label='β₂ (El Niño)')
ax1.axhline(0, color=C['normal'], lw=0.6)
ax1.set_xlabel('Cuantil τ')
ax1.set_ylabel('Coeficiente QR')
ax1.set_title('QR: coeficientes por cuantil\n(¿crece β₂ con τ?)', loc='left', fontsize=10)
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# P2 — Mean Excess Plot
ax2 = fig.add_subplot(gs[0, 1])
u_vals, e_u, n_u = mean_excess_plot(df.perdida_mS.values)
ax2.plot(u_vals, e_u, 'o-', color=C['mep'], lw=1.2, markersize=4, label='e(u) observado')
ax2.axvline(u_thresh, color=C['threshold'], lw=1.2, ls='--',
            label=f'Umbral u={u_thresh:.1f} M S/.')
# Ajuste lineal post-umbral para verificar GPD
mask_post = u_vals >= u_thresh
if mask_post.sum() > 2:
    slope_mep, intercept_mep, *_ = stats.linregress(u_vals[mask_post], e_u[mask_post])
    ax2.plot(u_vals[mask_post],
             intercept_mep + slope_mep*u_vals[mask_post],
             color=C['evt'], lw=1.2, ls='--', label=f'Ajuste lineal (ξ̂={xi_hat:.3f})')
ax2.set_xlabel('Umbral u (M S/.)')
ax2.set_ylabel('Exceso medio e(u)')
ax2.set_title('Mean Excess Plot\n(linealidad post-umbral valida GPD)', loc='left', fontsize=10)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# P3 — Densidad GPD vs. histograma de excesos
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(excesos, bins=25, density=True, color=C['fill'],
         edgecolor=C['loss'], lw=0.4, label='Excesos observados')
y_gpd = np.linspace(0, excesos.max()*1.1, 200)
if xi_hat != 0:
    pdf_gpd = (1/sigma_hat) * (1 + xi_hat*y_gpd/sigma_hat)**(-1/xi_hat - 1)
    pdf_gpd = np.where(1 + xi_hat*y_gpd/sigma_hat > 0, pdf_gpd, 0)
else:
    pdf_gpd = (1/sigma_hat) * np.exp(-y_gpd/sigma_hat)
ax3.plot(y_gpd, pdf_gpd, color=C['gpd'], lw=1.5, label=f'GPD(σ={sigma_hat:.2f}, ξ={xi_hat:.3f})')
ax3.set_xlabel('Exceso sobre umbral (M S/.)')
ax3.set_ylabel('Densidad')
ax3.set_title('GPD ajustada a excesos\n(cola pesada confirmada)', loc='left', fontsize=10)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

# P4 — Cuantiles: GPD vs. Normal vs. empírico
ax4 = fig.add_subplot(gs[1, 1])
p_exceedance = np.logspace(-3, -1, 40)  # de 0.001 a 0.1
q_gpd_range  = [gpd_quantile(u_thresh, sigma_hat, xi_hat, n_total, n_excess, p)
                for p in p_exceedance]
q_norm_range = [df.perdida_mS.mean() + stats.norm.ppf(1-p)*df.perdida_mS.std()
                for p in p_exceedance]

ax4.plot(1/p_exceedance, q_gpd_range,  color=C['evt'],    lw=1.5, label='GPD (EVT)')
ax4.plot(1/p_exceedance, q_norm_range, color=C['normal'], lw=1.5, ls='--', label='Normal')
ax4.axhline(df.perdida_mS.max(), color=C['loss'], lw=0.8, ls=':',
            label=f'Máx observado = {df.perdida_mS.max():.0f}')
ax4.set_xscale('log')
ax4.set_xlabel('Período de retorno (eventos)')
ax4.set_ylabel('Pérdida (M S/.)')
ax4.set_title('Nivel de retorno\n(GPD >> Normal en colas extremas)', loc='left', fontsize=10)
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# P5 — QR: percentiles condicionales por escenario
ax5 = fig.add_subplot(gs[2, :])
escenarios = {
    'Normal (no lluvia,\nno El Niño)'     : [0, 0],
    'Temporada lluvia\n(no El Niño)'      : [1, 0],
    'El Niño\n(+ temporada lluvia)'       : [1, 1],
}
x_esc = np.arange(len(escenarios))
width = 0.15
colors_tau = [C['normal'], '#60A5FA', '#2563EB', C['loss'], '#7F1D1D']

for idx, (tau, b) in enumerate(betas_qr.items()):
    q_vals = []
    for esc_name, (lluvia, nino) in escenarios.items():
        log_q = b[0] + b[1]*lluvia + b[2]*nino
        q_vals.append(np.exp(log_q))  # back-transform de log
    offset = (idx - 2) * width
    ax5.bar(x_esc + offset, q_vals, width=width,
            color=colors_tau[idx], alpha=0.85, label=f'τ={tau}')

ax5.set_xticks(x_esc)
ax5.set_xticklabels(list(escenarios.keys()), fontsize=9)
ax5.set_ylabel('Pérdida estimada (M S/.) — escala log')
ax5.set_yscale('log')
ax5.set_title('Panel 5 — Percentiles condicionales QR por escenario\n'
              '(cada barra = QR(τ) dado el escenario climático)', loc='left', fontsize=10)
ax5.legend(fontsize=8, loc='upper left')
ax5.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/finance_dashboard.png')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
df.to_csv('data/finance_indeci_losses.csv', index=False)
pd.DataFrame({'u': u_vals, 'e_u': e_u, 'n_u': n_u}).to_csv(
    'data/finance_mep.csv', index=False)
print('✓ data/finance_indeci_losses.csv')
print('✓ data/finance_mep.csv')
print('✓ data/finance_eda.png')
print('✓ data/finance_dashboard.png')

## Conclusiones — contexto INDECI / riesgo de desastres

| Concepto | En desastres naturales Perú | Aplicación Supply Chain |
|----------|---------------------------|-------------------------|
| **Función check** | Penaliza subestimar pérdidas >> sobreestimar | Penaliza stockout >> overstock |
| **β crece con τ** | El Niño impacta más percentiles extremos | Campaña impacta más cola de demanda |
| **ξ̂ > 0 (Fréchet)** | Pérdidas sin límite superior teórico | Demanda puede superar cualquier máximo histórico |
| **MEP lineal** | GPD válida sobre el umbral elegido | Validar antes de ajustar |
| **Nivel de retorno** | Pérdida esperada cada 1000 eventos | SS para fill rate 99.9% |

**Próximo:** `2_Supply_Adaptation.ipynb` — QR + EVT sobre errores de demanda Alicorp para calcular SS sin asumir normalidad.